In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: GEO01 - High-Risk Jurisdiction Operations
   
   BUSINESS QUESTION: 
   Does the Assessable Unit have operations within jurisdictions considered 
   high risk for bribery and corruption?
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. HR Enterprise List: hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGH-RISK JURISDICTIONS: Queries the td_country_risk_rating_abac table to isolate 
      countries where the FinalABACRating is strictly 'High'.
   3. HR ENTERPRISE LIST: Pulls ALL employees (not just contingent workers) from the 
      HR Enterprise List. Extracts CostCenterID and Workcountry. Applies the 
      canonical Cost Center normalization rule for string matching.
   4. HIGH-RISK FILTER: INNER JOINs the HR list to the High-Risk Countries table 
      to keep only employees working in high-risk jurisdictions.
   5. CC MAPPING: Maps cleaned Cost Centers to AU_IDs via the bootstrap view.
   6. FINAL OUTPUT: If an AU has ANY employee operating in a high-risk country, 
      the Response is 'Yes'. Otherwise 'No'. Default for unmapped AUs is 'No'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    -- 2. Grab jurisdictions strictly rated as 'High' risk
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE UPPER(TRIM(FinalABACRating)) = 'HIGH'
),

HR_Employees AS (
    -- 3. Pull ALL employees from the HR Enterprise List with canonical CC normalization
    SELECT 
        TRIM(CostCenterID) AS Raw_CC,
        CASE 
            WHEN TRIM(CostCenterID) RLIKE '^[0-9]+$' 
            THEN LPAD(RIGHT(TRIM(CostCenterID), 4), 4, '0')
            ELSE UPPER(TRIM(CostCenterID)) 
        END AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE CostCenterID IS NOT NULL
      AND TRIM(CostCenterID) != ''
),

High_Risk_Employees AS (
    -- 4. Keep only employees working in high-risk jurisdictions
    SELECT 
        e.Cleaned_CC,
        e.Workcountry
    FROM HR_Employees e
    INNER JOIN High_Risk_Countries h
        ON e.Workcountry = h.CountryName
),

CC_Mapping AS (
    -- 5. Standardize the CC Mapping table from our bootstrap
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(CAST(Cost_Center_ID AS STRING)) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CAST(Cost_Center_ID AS STRING)), 4), 4, '0') ELSE UPPER(TRIM(CAST(Cost_Center_ID AS STRING))) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND Cost_Center_ID IS NOT NULL
),

Flagged_AUs AS (
    -- 6. Map high-risk employees to AUs and count
    SELECT 
        m.AU_ID,
        COUNT(*) AS High_Risk_Employee_Count
    FROM High_Risk_Employees hr
    INNER JOIN CC_Mapping m
        ON hr.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

-- 7. Final Output: Yes/No anchored to the Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'GEO01' AS QuestionID,
    CASE 
        WHEN COALESCE(f.High_Risk_Employee_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: GEO01 - AU Level Calculation Review
   PURPOSE: One row per AU showing how the final Yes/No response was calculated,
   including the high-risk countries found and the cost centers that bridged.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE UPPER(TRIM(FinalABACRating)) = 'HIGH'
),

HR_Employees AS (
    SELECT 
        CASE 
            WHEN TRIM(CostCenterID) RLIKE '^[0-9]+$' 
            THEN LPAD(RIGHT(TRIM(CostCenterID), 4), 4, '0')
            ELSE UPPER(TRIM(CostCenterID)) 
        END AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE CostCenterID IS NOT NULL
      AND TRIM(CostCenterID) != ''
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(CAST(Cost_Center_ID AS STRING)) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CAST(Cost_Center_ID AS STRING)), 4), 4, '0') ELSE UPPER(TRIM(CAST(Cost_Center_ID AS STRING))) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND Cost_Center_ID IS NOT NULL
),

Mapped_All_Employees AS (
    -- All employees mapped to AUs (for total employee count per AU)
    SELECT 
        m.AU_ID,
        e.Cleaned_CC,
        e.Workcountry
    FROM HR_Employees e
    INNER JOIN CC_Mapping m
        ON e.Cleaned_CC = m.Mapped_CC
),

Mapped_High_Risk_Employees AS (
    -- Only high-risk employees mapped to AUs
    SELECT 
        m.AU_ID,
        e.Cleaned_CC,
        e.Workcountry
    FROM HR_Employees e
    INNER JOIN High_Risk_Countries h
        ON e.Workcountry = h.CountryName
    INNER JOIN CC_Mapping m
        ON e.Cleaned_CC = m.Mapped_CC
),

All_By_AU AS (
    SELECT 
        AU_ID,
        COUNT(*) AS Total_Employee_Count
    FROM Mapped_All_Employees
    GROUP BY AU_ID
),

High_Risk_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Workcountry))) AS High_Risk_Country_List,
        COUNT(*) AS High_Risk_Employee_Count
    FROM Mapped_High_Risk_Employees
    GROUP BY AU_ID
)

-- Output columns:
-- BusinessID: Master AU ID from the ABAC AU anchor list.
-- AU_Name: Master AU name tied to BusinessID.
-- Business_Segment: Business segment carried from the master AU list.
-- QuestionID: Questionnaire code for this debug output.
-- Response: Yes if any employee in a high-risk jurisdiction maps to this AU, otherwise No.
-- Normalized_CC_List: Normalized cost centers on the high-risk employee records.
-- High_Risk_Country_List: High-risk countries where employees mapped to this AU are working.
-- Total_Employee_Count: Total mapped employee rows for this AU.
-- High_Risk_Employee_Count: Mapped employees working in high-risk jurisdictions.
-- Why_Yes: Explanation provided when Response is Yes.
-- Why_No: Explanation provided when Response is No.
-- In_ABAC_AU_List: Confirms the AU came from the standard ABAC AU list.
SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'GEO01' AS QuestionID,
    CASE WHEN COALESCE(h.High_Risk_Employee_Count, 0) > 0 THEN 'Yes' ELSE 'No' END AS Response,
    COALESCE(h.Normalized_CC_List, 'n/a') AS Normalized_CC_List,
    COALESCE(h.High_Risk_Country_List, 'n/a') AS High_Risk_Country_List,
    COALESCE(a.Total_Employee_Count, 0) AS Total_Employee_Count,
    COALESCE(h.High_Risk_Employee_Count, 0) AS High_Risk_Employee_Count,
    CASE 
        WHEN COALESCE(h.High_Risk_Employee_Count, 0) > 0 
        THEN CONCAT('Yes because ', CAST(h.High_Risk_Employee_Count AS STRING), ' employee(s) in high-risk countries (', h.High_Risk_Country_List, ') mapped to this AU.')
        ELSE NULL
    END AS Why_Yes,
    CASE 
        WHEN COALESCE(h.High_Risk_Employee_Count, 0) = 0 AND COALESCE(a.Total_Employee_Count, 0) > 0 
        THEN CONCAT('No because ', CAST(a.Total_Employee_Count AS STRING), ' mapped employee(s) but none in a high-risk jurisdiction.')
        WHEN COALESCE(a.Total_Employee_Count, 0) = 0 
        THEN 'No because no HR records mapped to this AU through the CC bootstrap.'
        ELSE NULL
    END AS Why_No,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN High_Risk_By_AU h
    ON mast.BusinessID = h.AU_ID
LEFT JOIN All_By_AU a
    ON mast.BusinessID = a.AU_ID
ORDER BY mast.BusinessID;
